# FightPredict: Machine Learning Only

This notebook contains the complete ML workflow for predicting whether fighter A wins a UFC bout. It uses only pre-fight feature differences and a chronological evaluation design to avoid training on future fights.

In [ ]:
from pathlib import Path
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay, accuracy_score, brier_score_loss,
    f1_score, log_loss, roc_auc_score, RocCurveDisplay
)
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
pd.set_option('display.max_columns', 30)

## 1. Load the prepared dataset

Each row is one completed bout. `winner_a` is the binary target. Model inputs are differences between fighter A and fighter B; names and bout metadata are not used as features.

In [ ]:
DATA_PATH = Path('data/processed_bouts.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('ml_only/data/processed_bouts.csv')  # running from repository root

df = pd.read_csv(DATA_PATH, parse_dates=['date'])
FEATURES = [
    'age_diff', 'height_cm_diff', 'reach_cm_diff', 'weight_kg_diff',
    'wins_diff', 'losses_diff', 'win_streak_diff', 'lose_streak_diff',
    'striking_accuracy_diff', 'takedown_accuracy_diff',
    'experience_diff', 'smoothed_win_rate_diff'
]
TARGET = 'winner_a'
print(f'{len(df):,} bouts from {df.date.min().date()} to {df.date.max().date()}')
df.head()

In [ ]:
summary = pd.DataFrame({
    'dtype': df[FEATURES].dtypes.astype(str),
    'missing_%': (df[FEATURES].isna().mean() * 100).round(2),
    'mean': df[FEATURES].mean().round(3),
    'std': df[FEATURES].std().round(3),
})
display(summary)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
df.groupby(df.date.dt.year).size().plot.bar(ax=axes[0], color='#7357f5', title='Bouts by year')
(df[FEATURES].isna().mean() * 100).sort_values().plot.barh(
    ax=axes[1], color='#7357f5', title='Missing values (%)'
)
plt.tight_layout();

## 2. Chronological split

A random split would allow old fights to be evaluated using patterns learned from future fights. Instead, model selection uses 2023, calibration uses 2024, and 2025+ remains untouched until final evaluation.

In [ ]:
masks = {
    'train': df.date < '2023-01-01',
    'validation': (df.date >= '2023-01-01') & (df.date < '2024-01-01'),
    'calibration': (df.date >= '2024-01-01') & (df.date < '2025-01-01'),
    'test': df.date >= '2025-01-01',
}

split_summary = []
for name, mask in masks.items():
    split_summary.append({
        'split': name, 'bouts': int(mask.sum()),
        'start': df.loc[mask, 'date'].min().date(),
        'end': df.loc[mask, 'date'].max().date(),
        'fighter_A_win_rate': df.loc[mask, TARGET].mean(),
    })
pd.DataFrame(split_summary).set_index('split')

## 3. Preprocessing and symmetry

Missing values are imputed with medians learned from the training data, then features are standardized. Training examples are mirrored by negating all A-minus-B features and reversing the target. Prediction also averages both orientations, guaranteeing `P(A beats B) = 1 - P(B beats A)`.

In [ ]:
def make_pipeline(estimator):
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median', keep_empty_features=True)),
        ('scaler', StandardScaler()),
        ('classifier', estimator),
    ])

def augment(X, y):
    return pd.concat([X, -X], ignore_index=True), np.concatenate([y, 1 - y])

def symmetric_probability(model, X):
    forward = model.predict_proba(X)[:, 1]
    reverse = model.predict_proba(-X)[:, 1]
    return (forward + 1 - reverse) / 2

def evaluate(y_true, probability):
    prediction = probability >= 0.5
    return {
        'accuracy': accuracy_score(y_true, prediction),
        'roc_auc': roc_auc_score(y_true, probability),
        'f1': f1_score(y_true, prediction),
        'brier': brier_score_loss(y_true, probability),
        'log_loss': log_loss(y_true, probability),
    }

X = df[FEATURES]
y = df[TARGET].to_numpy()
X_train, y_train = augment(X.loc[masks['train']], y[masks['train']])
X_validation, y_validation = X.loc[masks['validation']], y[masks['validation']]

## 4. Train and select models

Six configurations are compared using validation Brier score. Lower Brier score means better probability predictions. The test set is not used for this choice.

In [ ]:
configurations = {
    'Logistic Regression C=0.1': LogisticRegression(C=0.1, max_iter=1200, random_state=SEED),
    'Logistic Regression C=1': LogisticRegression(C=1, max_iter=1200, random_state=SEED),
    'Random Forest leaf=12': RandomForestClassifier(
        n_estimators=350, min_samples_leaf=12, max_depth=10, n_jobs=-1, random_state=SEED
    ),
    'Random Forest leaf=25': RandomForestClassifier(
        n_estimators=350, min_samples_leaf=25, max_depth=8, n_jobs=-1, random_state=SEED
    ),
    'Neural Network (32, 16)': MLPClassifier(
        hidden_layer_sizes=(32, 16), alpha=1, max_iter=350, random_state=SEED
    ),
    'Neural Network (16,)': MLPClassifier(
        hidden_layer_sizes=(16,), alpha=3, max_iter=350, random_state=SEED
    ),
}

models, validation_rows = {}, []
for name, estimator in configurations.items():
    model = make_pipeline(estimator)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', ConvergenceWarning)
        model.fit(X_train, y_train)
    probability = symmetric_probability(model, X_validation)
    models[name] = model
    validation_rows.append({'model': name, **evaluate(y_validation, probability)})

validation_results = pd.DataFrame(validation_rows).sort_values('brier').reset_index(drop=True)
best_name = validation_results.loc[0, 'model']
best_model = models[best_name]
print('Selected model:', best_name)
validation_results.round(4)

In [ ]:
validation_results.sort_values('brier').plot.barh(
    x='model', y='brier', figsize=(9, 4), color='#7357f5', legend=False,
    title='Validation Brier score (lower is better)'
)
plt.axvline(0.25, color='gray', linestyle='--', label='50/50 baseline')
plt.xlabel('Brier score'); plt.legend(); plt.tight_layout();

## 5. Calibrate probabilities on 2024

A single slope is fitted to the selected model's logits. No intercept is used, preserving the swap-symmetry property.

In [ ]:
X_calibration = X.loc[masks['calibration']]
y_calibration = y[masks['calibration']]
p_calibration = symmetric_probability(best_model, X_calibration)
logits = np.log(np.clip(p_calibration, 1e-6, 1 - 1e-6) / np.clip(1 - p_calibration, 1e-6, 1 - 1e-6))

calibrator = LogisticRegression(fit_intercept=False, C=10, max_iter=1200, random_state=SEED)
calibrator.fit(
    np.concatenate([logits, -logits]).reshape(-1, 1),
    np.concatenate([y_calibration, 1 - y_calibration]),
)
calibration_slope = max(0.0, float(calibrator.coef_[0, 0]))

def apply_calibration(probability, slope=calibration_slope):
    p = np.clip(np.asarray(probability), 1e-6, 1 - 1e-6)
    z = np.clip(slope * np.log(p / (1 - p)), -35, 35)
    return 1 / (1 + np.exp(-z))

print(f'Calibration slope: {calibration_slope:.4f}')
pd.DataFrame([
    {'version': 'before calibration', **evaluate(y_calibration, p_calibration)},
    {'version': 'after calibration', **evaluate(y_calibration, apply_calibration(p_calibration))},
]).set_index('version').round(4)

## 6. Final test evaluation

This is the first use of the 2025+ test labels. These results estimate performance on later fights and must not be used to retune the model.

In [ ]:
X_test = X.loc[masks['test']]
y_test = y[masks['test']]
p_test = apply_calibration(symmetric_probability(best_model, X_test))
test_metrics = evaluate(y_test, p_test)
pd.Series(test_metrics, name='test').to_frame().round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
RocCurveDisplay.from_predictions(y_test, p_test, ax=axes[0], color='#7357f5')
axes[0].plot([0, 1], [0, 1], '--', color='gray')
axes[0].set_title('Chronological test ROC curve')
ConfusionMatrixDisplay.from_predictions(
    y_test, p_test >= 0.5, display_labels=['Fighter B', 'Fighter A'],
    cmap='Purples', ax=axes[1], colorbar=False
)
axes[1].set_title('Test confusion matrix')
plt.tight_layout();

## 7. Feature importance

Permutation importance is measured on validation data, keeping the final test set separate from interpretation and selection.

In [ ]:
rng = np.random.default_rng(SEED)
base_auc = roc_auc_score(y_validation, symmetric_probability(best_model, X_validation))
importance_rows = []
for feature in FEATURES:
    drops = []
    for _ in range(5):
        shuffled = X_validation.copy()
        shuffled[feature] = rng.permutation(shuffled[feature].to_numpy())
        shuffled_auc = roc_auc_score(y_validation, symmetric_probability(best_model, shuffled))
        drops.append(base_auc - shuffled_auc)
    importance_rows.append({'feature': feature, 'auc_drop': np.mean(drops), 'std': np.std(drops)})

importance = pd.DataFrame(importance_rows).sort_values('auc_drop')
importance.plot.barh(x='feature', y='auc_drop', xerr='std', figsize=(9, 5), color='#7357f5', legend=False)
plt.xlabel('Validation AUC decrease after shuffling')
plt.title('Permutation feature importance')
plt.tight_layout()

## 8. Save the trained model

The saved bundle contains the full preprocessing pipeline, model, feature order, and calibration slope. Only load joblib files from trusted sources.

In [ ]:
artifact = {
    'pipeline': best_model,
    'calibration_slope': calibration_slope,
    'feature_names': FEATURES,
    'selected_model': best_name,
    'test_metrics': test_metrics,
}
joblib.dump(artifact, 'fightpredict_ml_model.joblib', compress=3)
print('Saved fightpredict_ml_model.joblib')